# Weird stats — the modern era (last 20 seasons)

Hunting the oddities that make a good Short: **true, checkable, and surprising**.

Read-only throughout — `sportsdb` attaches every SQLite file `READ_ONLY` through
DuckDB, so nothing here can touch the source databases.

## Four guards, carried over from `superlative`

Each of these caught a real wrong answer. They are one-liners here so you do not
rediscover them interactively at 1am:

| Guard | Why |
|---|---|
| `WHERE season_type = 'Regular Season'` | Pooling playoffs turned Curry's record 402 threes into **482** |
| `PTS_EXPR` for play-by-play scoring | Summing `shot_value` gave a 45-point game **32** — free throws carry 0 |
| Per-column floors | `fga` is non-zero from 1946-47 but 91% NULL — a rate over it reported FG% of **1.938** |
| Rate minimums | "Best FG%" unqualified returns whoever went 1-for-1 |

Rule of thumb for anything you plan to publish: **if it is a rate, it needs a
minimum and a completeness floor. If it is from play-by-play, it cannot say
"ever" — that table starts in 1996-97.**

In [ ]:
import sportsdb

con = sportsdb.connect()
sportsdb.databases()

In [ ]:
# ---- era + guards -----------------------------------------------------------

# "Last 20 years". Change once here; every query below reads it.
ERA_START = "2005-06"
REGULAR = "Regular Season"

# The verified points expression. Free-throw rows carry shot_value = 0 and are
# distinguished from misses only by a MISS prefix, so summing shot_value
# undercounts every trip to the line. Reconciled against player_game.pts over
# 1369 player-games with zero mismatches.
PTS_EXPR = """
    CASE
        WHEN action_type = 'Made Shot' THEN shot_value
        WHEN action_type = 'Free Throw' AND description NOT LIKE 'MISS%' THEN 1
        ELSE 0
    END
"""

# Coverage floors, measured rather than assumed. Anything ranked on a column
# cannot speak before the season that column started being recorded.
sportsdb.q(f"""
SELECT 'pts' col, MIN(season) first_nonzero FROM nba.player_game WHERE pts > 0
UNION ALL SELECT 'fg3m', MIN(season) FROM nba.player_game WHERE fg3m > 0
UNION ALL SELECT 'blk',  MIN(season) FROM nba.player_game WHERE blk  > 0
UNION ALL SELECT 'stl',  MIN(season) FROM nba.player_game WHERE stl  > 0
UNION ALL SELECT 'tov',  MIN(season) FROM nba.player_game WHERE tov  > 0
UNION ALL SELECT 'plus_minus', MIN(season) FROM nba.player_game WHERE plus_minus <> 0
ORDER BY 2
""")

In [ ]:
# ---- helpers ---------------------------------------------------------------
import json
from pathlib import Path


def era(alias: str = "pg") -> str:
    """The standard era + season-type filter. Use in every query."""
    return f"{alias}.season >= '{ERA_START}' AND {alias}.season_type = '{REGULAR}'"


def is_it_real(sql: str, expect_rows: int | None = None):
    """Run a candidate and eyeball it before believing it.

    Every wrong answer this stack produced looked plausible. Print the rows,
    check one against a source you trust, THEN write the Short.
    """
    df = sportsdb.q(sql)
    print(f"{len(df)} rows")
    if expect_rows is not None and len(df) != expect_rows:
        print(f"  !! expected {expect_rows} rows - the query is not saying what you think")
    return df


def to_block_json(df, title: str, note: str, hook: str, value_col: str,
                  label_col: str = "player_name", out_dir: str = "outputs"):
    """Write a finding as a stat-blocks factsheet, ready to render.

    `note` is mandatory and is the scope you are entitled to claim - not a
    caption. If you cannot state the scope, the finding is not ready.
    """
    if not note.strip():
        raise ValueError("a finding without its scope is not publishable")
    facts = [
        {
            "id": f"nb.{title.lower().replace(' ', '-')}.{i}",
            "kind": "notebook-finding",
            "text": f"{i}. {row[label_col]} - {row[value_col]}",
            "data": {"rank": i, "name": str(row[label_col]),
                     "value": float(row[value_col]), "tied": False},
            "team": None,
            "source": f"nba via analysis/02_weird_stats_modern.ipynb; scope {note}",
        }
        for i, (_, row) in enumerate(df.iterrows(), start=1)
    ]
    path = Path(out_dir) / f"finding-{title.lower().replace(' ', '-')}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(
        {"league": "NBA", "label": title, "hook": hook, "facts": facts},
        indent=2), encoding="utf-8")
    print(f"wrote {path}")
    print("render it:  cd ../../stat-blocks && python -c \"...build.from_factsheet(...)\"")
    return path

---
## Hunt 1 — 40-point games in a loss

The shape that makes a good Short: a huge individual number attached to a bad
outcome. Tension built in, and the viewer can't guess it.

In [ ]:
is_it_real(f"""
SELECT p.player_name, pg.pts, pg.season, pg.game_date, pg.matchup
FROM nba.player_game pg
JOIN nba.players p ON p.player_id = pg.player_id
WHERE {era()} AND pg.wl = 'L' AND pg.pts >= 50
ORDER BY pg.pts DESC
LIMIT 15
""")

---
## Hunt 2 — the one-hit wonders

Players whose single best game towers over their career average. "He did this
once and never again" is a story; a leaderboard is not.

In [ ]:
is_it_real(f"""
WITH career AS (
    SELECT pg.player_id, COUNT(*) games, AVG(pg.pts) ppg, MAX(pg.pts) best
    FROM nba.player_game pg
    WHERE {era()}
    GROUP BY pg.player_id
    HAVING COUNT(*) >= 200 AND AVG(pg.pts) >= 4
)
SELECT p.player_name, c.games, ROUND(c.ppg, 1) ppg, c.best,
       ROUND(c.best / c.ppg, 1) AS times_his_average
FROM career c JOIN nba.players p ON p.player_id = c.player_id
ORDER BY times_his_average DESC
LIMIT 15
""")

---
## Hunt 3 — draft steals

The `drafts` table joins to everything. Second-round picks who out-produced
their draft slot is a format the sleep-sports factsheets already use.

In [ ]:
is_it_real(f"""
WITH production AS (
    SELECT pg.player_id, SUM(pg.pts) career_pts, COUNT(*) games
    FROM nba.player_game pg
    WHERE pg.season_type = '{REGULAR}'
    GROUP BY pg.player_id
)
SELECT d.player_name, d.season AS draft_year, d.overall_pick,
       d.round_number, pr.games, pr.career_pts
FROM nba.drafts d
JOIN production pr ON pr.player_id = d.person_id
WHERE d.round_number >= 2 AND CAST(SUBSTR(d.season, 1, 4) AS INT) >= 2005
ORDER BY pr.career_pts DESC
LIMIT 15
""")

---
## Hunt 4 — losing while shooting better

A team-level oddity: won the efficiency battle and lost the game. Uses
`team_game`, and the self-join on `game_id` gives you both sides.

In [ ]:
is_it_real(f"""
SELECT t.season, t.game_date, t.matchup,
       ROUND(t.fg_pct, 3) loser_fg, ROUND(o.fg_pct, 3) winner_fg,
       t.pts loser_pts, o.pts winner_pts,
       ROUND(t.fg_pct - o.fg_pct, 3) AS fg_edge_wasted
FROM nba.team_game t
JOIN nba.team_game o ON o.game_id = t.game_id AND o.team_id <> t.team_id
WHERE t.season >= '{ERA_START}' AND t.season_type = '{REGULAR}'
  AND t.wl = 'L' AND t.fga > 0 AND o.fga > 0
ORDER BY fg_edge_wasted DESC
LIMIT 15
""")

---
## Hunt 5 — the quarter nobody survived (play-by-play)

**Scope warning:** this reads `play_by_play`, which starts in **1996-97**. A
claim from here can never say "ever". Note the `PTS_EXPR` — do not sum
`shot_value`.

In [ ]:
is_it_real(f"""
WITH quarters AS (
    SELECT pbp.game_id, pbp.period, pbp.person_id,
           SUM({PTS_EXPR}) AS pts
    FROM nba.play_by_play pbp
    JOIN nba.games g ON g.game_id = pbp.game_id
    WHERE g.season >= '{ERA_START}' AND g.season_type = '{REGULAR}'
    GROUP BY pbp.game_id, pbp.period, pbp.person_id
)
SELECT p.player_name, q.pts AS points_in_quarter, q.period, g.season, g.game_date
FROM quarters q
JOIN nba.players p ON p.player_id = q.person_id
JOIN nba.games g ON g.game_id = q.game_id
WHERE q.pts >= 25
ORDER BY q.pts DESC
LIMIT 15
""")

---
## Hunt 6 — your own

Scratch space. Before turning anything here into a Short:

1. **Check one row against a source you trust.** Every wrong answer this stack
   produced looked plausible — a 1.938 field-goal percentage, Curry with 482
   threes, a 45-point game totalling 32.
2. **State the scope out loud.** If you cannot say "since when", the finding is
   not ready.
3. **Watch for a player appearing twice.** A per-game leaderboard lets one name
   fill several slots (the blocks top-5 is Elmore Smith, Bol, Bol, Shaq, Smith),
   which is correct data that reads as a bug. `GROUP BY player_id` with a `MAX`
   if you want distinct names.
4. `to_block_json(df, title, note, hook, value_col)` when it survives all three.

In [ ]:
sportsdb.tables("nba")